In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import f_oneway

PROJECT_ROOT = Path.cwd().parents[2]
sys.path.insert(0, str(PROJECT_ROOT))

from src.feature_selection.data_loading import load_split


In [2]:
# ANOVA must only ever see the train split.
# (Previously this loaded ml_dataset.csv (train+validation+test)
# and a later cell silently re-derived X, y from that full
# dataframe, discarding the train-only split below -> test leakage.)
X, y = load_split("train", processed_dir=PROJECT_ROOT / "data" / "processed")

print("X shape:", X.shape)
print("y shape:", y.shape)


X shape: (1895, 25)
y shape: (1895,)


In [3]:
# feature별 고유값 개수 확인
feature_info = pd.DataFrame({
    "feature": X.columns,
    "unique_count": X.nunique(),
    "missing_count": X.isna().sum(),
})

feature_info


,feature,unique_count,missing_count
return_1d,return_1d,1743,0
return_5d,return_5d,1800,0
return_10d,return_10d,1838,0
return_20d,return_20d,1840,0
intraday_return,intraday_return,1692,0
high_low_range,high_low_range,1661,0
gap,gap,1580,0
sma_5,sma_5,1553,0
sma_20,sma_20,1806,0
sma_60,sma_60,1863,0


In [4]:
X.dtypes


return_1d           float64
return_5d           float64
return_10d          float64
return_20d          float64
intraday_return     float64
high_low_range      float64
gap                 float64
sma_5               float64
sma_20              float64
sma_60              float64
price_to_sma_5      float64
price_to_sma_20     float64
price_to_sma_60     float64
rsi_14              float64
roc_10              float64
roc_20              float64
macd                float64
macd_signal         float64
macd_hist           float64
volatility_5        float64
volatility_20       float64
atr_14              float64
volume_change_1d    float64
volume_sma_20       float64
volume_ratio_20     float64
dtype: object

In [5]:
feature_cols = X.columns.tolist()

print("Feature 개수:", len(feature_cols))
print("Feature:", feature_cols)


Feature 개수: 25
Feature: ['return_1d', 'return_5d', 'return_10d', 'return_20d', 'intraday_return', 'high_low_range', 'gap', 'sma_5', 'sma_20', 'sma_60', 'price_to_sma_5', 'price_to_sma_20', 'price_to_sma_60', 'rsi_14', 'roc_10', 'roc_20', 'macd', 'macd_signal', 'macd_hist', 'volatility_5', 'volatility_20', 'atr_14', 'volume_change_1d', 'volume_sma_20', 'volume_ratio_20']


In [6]:
anova_results = []

for feature in feature_cols:
    temp = pd.DataFrame({
        "feature": X[feature],
        "target": y
    }).dropna()

    # feature 값을 4개 분위수 그룹으로 분할
    temp["group"] = pd.qcut(
        temp["feature"],
        q=4,
        labels=False,
        duplicates="drop"
    )

    # 그룹별 target 추출
    groups = [
        group["target"].values
        for _, group in temp.groupby("group", observed=True)
    ]

    # 그룹이 2개 이상일 때만 ANOVA
    if len(groups) >= 2:
        f_stat, p_value = f_oneway(*groups)

        anova_results.append({
            "feature": feature,
            "f_statistic": f_stat,
            "p_value": p_value
        })

anova_df = pd.DataFrame(anova_results)

anova_df


,feature,f_statistic,p_value
0,return_1d,0.130465,0.942001
1,return_5d,2.826510,0.037355
2,return_10d,2.165305,0.090197
3,return_20d,6.126449,0.000382
4,intraday_return,0.795305,0.496425
5,high_low_range,1.174695,0.317954
6,gap,2.308245,0.074693
7,sma_5,0.600930,0.614411
8,sma_20,0.545555,0.651167
9,sma_60,1.099382,0.348180


In [7]:
anova_df = anova_df.sort_values("p_value")

anova_df


,feature,f_statistic,p_value
18,macd_hist,7.032139,0.000106
20,volatility_20,6.484064,0.000231
3,return_20d,6.126449,0.000382
15,roc_20,6.126449,0.000382
11,price_to_sma_20,5.076592,0.001675
13,rsi_14,5.036390,0.001772
1,return_5d,2.826510,0.037355
24,volume_ratio_20,2.524846,0.056002
6,gap,2.308245,0.074693
16,macd,2.231634,0.082651


In [8]:
feature = "rsi_14"

temp = pd.DataFrame({
    "feature": X[feature],
    "target": y
}).dropna()

temp["group"] = pd.qcut(
    temp["feature"],
    q=4,
    labels=["Q1", "Q2", "Q3", "Q4"],
    duplicates="drop"
)

print(temp.groupby("group", observed=True)["target"].agg(["count", "mean", "std"]))


       count      mean       std
group                           
Q1       474  0.006016  0.055069
Q2       474  0.008693  0.053147
Q3       473  0.003113  0.059312
Q4       474  0.018230  0.082364


In [9]:
groups = [
    temp.loc[temp["group"] == group, "target"].values
    for group in ["Q1", "Q2", "Q3", "Q4"]
]

f_stat, p_value = f_oneway(*groups)

print("F-statistic:", f_stat)
print("p-value:", p_value)


F-statistic: 5.036389955907839
p-value: 0.001771825968798029


In [10]:
temp.groupby("group", observed=True)["target"].mean()


group
Q1    0.006016
Q2    0.008693
Q3    0.003113
Q4    0.018230
Name: target, dtype: float64

In [11]:
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed" / "filter_results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

anova_df.to_csv(
    OUTPUT_DIR / "anova_target.csv",
    index=False,
)
